# 3.0 How does one trust compare with its peers?

**Board question:** where does one trust sit within its own sector, and against trusts of its own size?

The analysis is done in SQL. Every query lives in
[`sql/analysis/03_peer_benchmarking.sql`](../../sql/analysis/03_peer_benchmarking.sql) and runs against the MySQL warehouse
(`nhs_gold` views and the `nhs_silver.fct_tac` fact table). Each section below shows the query exactly as
it is in that file, then its result. Python only runs the query and, in one place, draws a chart from the
result.

| Query | SQL technique |
| --- | --- |
| `choose_trust` | Session variable as a parameter, defaulted with `COALESCE` |
| `peer_position` | Unpivot with `UNION ALL`, then `PERCENT_RANK()`, `NTILE()`, median |
| `nearest_peers` | CTE self-join, `ROW_NUMBER()` ordered by distance |
| `trend_vs_sector` | Two medians in one pass, `LAG()` for the trust's change |
| `underlying_result` | Several SubCodes pivoted from the fact table in one pass |

**Data:** NHS England Trust Accounts Consolidation (TAC), annual and actuals only. Money is held in £000s;
columns ending `_m` are £m. Operating surplus is the SoCI line `SCI0140A` as each trust reported it, which
includes impairments (non-cash write-downs). No query is pinned to a year: each one reads the year range
from the data.

To run it: load the warehouse, `export DB_PASSWORD=...`, then run all cells.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Make analysis/nhs_analysis.py importable from the repo root or from this folder
here = Path.cwd()
sys.path.insert(0, str(here.parent if here.name == "notebooks" else here / "analysis"))
import nhs_analysis as nhs

nhs.set_style()
pd.options.display.float_format = "{:,.1f}".format
pd.options.display.max_columns = 20
import os

## 3.1 Choose the trust

The trust is a parameter, never written into a query: the SQL reads the session variable `@org_code`. Set
the `TRUST_ORG_CODE` environment variable to an ODS code to choose one. Left unset, the query defaults to
the trust with the lowest operating margin in the latest year.

In [2]:
org = os.environ.get("TRUST_ORG_CODE")
trust = nhs.sql("03_peer_benchmarking", "choose_trust", **({"org_code": org} if org else {}))
trust

`sql/analysis/03_peer_benchmarking.sql` → **choose_trust**

```sql
-- Technique: a session variable as the query parameter, defaulted with COALESCE
-- and a subquery when the caller has not set it. A variable set in the client
-- takes the connection's collation, which can differ from the tables'
-- (utf8mb4_unicode_ci); converting it once here stops "Illegal mix of
-- collations" errors in every later comparison with org_code.
SET @org_code = CONVERT(COALESCE(
    @org_code,
    (SELECT org_code
     FROM v_kpis
     WHERE financial_year = (SELECT MAX(financial_year) FROM nhs_silver.fct_tac)
     ORDER BY operating_surplus_000s / NULLIF(total_income_000s, 0)
     LIMIT 1)
) USING utf8mb4) COLLATE utf8mb4_unicode_ci;
SELECT org_code, organisation_name, sector, region, trust_type
FROM v_kpis
WHERE org_code = @org_code
  AND financial_year = (SELECT MAX(financial_year) FROM nhs_silver.fct_tac);
```

,org_code,organisation_name,sector,region,trust_type
0,RGR,West Suffolk NHS Foundation Trust,Acute,East of England,FOUNDATION_TRUST


## 3.2 Position within the sector

Five measures are unpivoted into rows with `UNION ALL`, so a single set of window functions ranks them
all. `PERCENT_RANK()` gives the percentile (100 = highest in the sector), `NTILE(4)` the quartile, and the
`ROW_NUMBER()` median gives the comparison point.

In [3]:
nhs.sql("03_peer_benchmarking", "peer_position")

`sql/analysis/03_peer_benchmarking.sql` → **peer_position**

```sql
-- Where the trust sits in its sector on five measures in the latest year.
-- percentile = 100 for the highest value in the sector, 0 for the lowest.
-- Technique: unpivot the measures to rows with UNION ALL, then PERCENT_RANK(),
-- a sector median and a quartile with NTILE(), all as window functions over
-- the same partition.
WITH latest AS (
    SELECT MAX(financial_year) AS financial_year FROM nhs_silver.fct_tac
),
peers AS (
    SELECT s.*
    FROM v_kpis s
    JOIN latest l ON l.financial_year = s.financial_year
    WHERE s.sector = (SELECT sector FROM nhs_silver.dim_trust WHERE org_code = @org_code)
),
measures AS (
    SELECT org_code, 1 AS ord, 'Operating margin %' AS measure,
           100 * operating_surplus_000s / NULLIF(total_income_000s, 0) AS value FROM peers
    UNION ALL SELECT org_code, 2, 'EBITDA margin %',       ebitda_margin_pct FROM peers
    UNION ALL SELECT org_code, 3, 'Pay as % of income',    pay_pct_income    FROM peers
    UNION ALL SELECT org_code, 4, 'Net surplus margin %',  net_surplus_margin_pct FROM peers
    UNION ALL SELECT org_code, 5, 'Income £m',             total_income_000s / 1000 FROM peers
),
ranked AS (
    SELECT
        org_code, ord, measure, value,
        PERCENT_RANK() OVER w                                           AS pct_rank,
        NTILE(4)       OVER w                                           AS quartile,
        ROW_NUMBER()   OVER w                                           AS rn,
        COUNT(*)       OVER (PARTITION BY measure)                      AS n
    FROM measures
    WHERE value IS NOT NULL
    WINDOW w AS (PARTITION BY measure ORDER BY value)
),
medians AS (
    SELECT measure, AVG(CASE WHEN rn IN (FLOOR((n + 1) / 2), CEIL((n + 1) / 2)) THEN value END) AS sector_median
    FROM ranked
    GROUP BY measure
)
SELECT
    r.measure,
    ROUND(r.value, 1)              AS trust_value,
    ROUND(m.sector_median, 1)      AS sector_median,
    ROUND(r.value - m.sector_median, 1) AS difference,
    ROUND(100 * r.pct_rank, 0)     AS percentile,
    r.quartile                     AS quartile_1_low_4_high,
    r.n                            AS trusts_in_sector
FROM ranked r
JOIN medians m ON m.measure = r.measure
WHERE r.org_code = @org_code
ORDER BY r.ord;
```

,measure,trust_value,sector_median,difference,percentile,quartile_1_low_4_high,trusts_in_sector
0,Operating margin %,-22.9,-0.8,-22.2,0,1,118
1,EBITDA margin %,3.4,3.8,-0.4,40,2,118
2,Pay as % of income,69.7,64.8,5.0,79,4,118
3,Net surplus margin %,-24.2,-2.9,-21.3,1,1,118
4,Income £m,397.3,683.6,-286.3,16,1,118


## 3.3 Nearest peers by size

Comparing a mid-sized trust with the whole sector mixes it with trusts several times larger. This picks
the five trusts in the same sector closest to it in income: the CTE is joined to itself so each row
carries the selected trust's income, and `ROW_NUMBER()` orders peers by the distance between the two.

In [4]:
nhs.sql("03_peer_benchmarking", "nearest_peers")

`sql/analysis/03_peer_benchmarking.sql` → **nearest_peers**

```sql
-- A fairer peer group than the whole sector: the five trusts in the same sector
-- closest in size (income) to the selected trust, and how they performed.
-- Technique: a CTE self-joined so every peer row carries the selected trust's
-- income, ROW_NUMBER() ordered by the distance between the two, and the selected
-- trust kept in the output (distance 0) for comparison.
WITH sector_latest AS (
    SELECT s.org_code, s.organisation_name, s.total_income_000s, s.operating_surplus_000s,
           s.ebitda_margin_pct, s.pay_pct_income
    FROM v_kpis s
    WHERE s.financial_year = (SELECT MAX(financial_year) FROM nhs_silver.fct_tac)
      AND s.sector = (SELECT sector FROM nhs_silver.dim_trust WHERE org_code = @org_code)
),
candidates AS (
    SELECT
        p.*,
        ROW_NUMBER() OVER (ORDER BY ABS(p.total_income_000s - t.total_income_000s)) - 1 AS distance_rank
    FROM sector_latest p
    JOIN sector_latest t ON t.org_code = @org_code
)
SELECT
    IF(org_code = @org_code, 'Selected trust', CONCAT('Peer ', distance_rank)) AS role,
    organisation_name,
    ROUND(total_income_000s / 1000, 1)                                          AS income_m,
    ROUND(100 * operating_surplus_000s / NULLIF(total_income_000s, 0), 1)       AS operating_margin_pct,
    ebitda_margin_pct,
    pay_pct_income
FROM candidates
WHERE distance_rank <= 5
ORDER BY distance_rank;
```

,role,organisation_name,income_m,operating_margin_pct,ebitda_margin_pct,pay_pct_income
0,Selected trust,West Suffolk NHS Foundation Trust,397.3,-22.9,3.4,69.7
1,Peer 1,Mid Cheshire Hospitals NHS Foundation Trust,399.9,-13.8,-2.2,72.8
2,Peer 2,Gateshead Health NHS Foundation Trust,401.1,-1.5,1.3,67.3
3,Peer 3,Salisbury NHS Foundation Trust,384.8,2.9,6.9,62.9
4,Peer 4,The Princess Alexandra Hospital NHS Trust,383.8,-1.7,2.9,68.6
5,Peer 5,Chesterfield Royal Hospital NHS Foundation Trust,381.7,-6.4,-0.7,69.9


## 3.4 Trend against the sector median

Both sector medians (operating margin and pay share) are computed in one pass with two `ROW_NUMBER()`
orderings over the same partition, then the trust is joined onto its own sector.

In [5]:
nhs.sql("03_peer_benchmarking", "trend_vs_sector")

`sql/analysis/03_peer_benchmarking.sql` → **trend_vs_sector**

```sql
-- The trust against the median trust in its sector, every year, for operating
-- margin and pay as a share of income.
-- Technique: medians for every sector-year in one pass, a LEFT JOIN of the trust
-- onto its own sector, and LAG() for the trust's year-on-year change.
WITH base AS (
    SELECT org_code, sector, financial_year,
           100 * operating_surplus_000s / NULLIF(total_income_000s, 0) AS op_margin,
           pay_pct_income
    FROM v_kpis
),
ranked AS (
    SELECT sector, financial_year, op_margin, pay_pct_income,
           ROW_NUMBER() OVER (PARTITION BY sector, financial_year ORDER BY op_margin)      AS rn_m,
           ROW_NUMBER() OVER (PARTITION BY sector, financial_year ORDER BY pay_pct_income) AS rn_p,
           COUNT(*)     OVER (PARTITION BY sector, financial_year)                         AS n
    FROM base
),
medians AS (
    SELECT sector, financial_year,
           AVG(CASE WHEN rn_m IN (FLOOR((n + 1) / 2), CEIL((n + 1) / 2)) THEN op_margin END)      AS median_op_margin,
           AVG(CASE WHEN rn_p IN (FLOOR((n + 1) / 2), CEIL((n + 1) / 2)) THEN pay_pct_income END) AS median_pay_pct
    FROM ranked
    GROUP BY sector, financial_year
)
SELECT
    m.financial_year,
    ROUND(t.op_margin, 1)                                                AS trust_op_margin_pct,
    ROUND(m.median_op_margin, 1)                                         AS sector_median_op_margin_pct,
    ROUND(t.op_margin - LAG(t.op_margin) OVER (ORDER BY m.financial_year), 1) AS trust_change_pts,
    ROUND(t.pay_pct_income, 1)                                           AS trust_pay_pct,
    ROUND(m.median_pay_pct, 1)                                           AS sector_median_pay_pct
FROM medians m
JOIN base t
  ON t.sector = m.sector AND t.financial_year = m.financial_year AND t.org_code = @org_code
ORDER BY m.financial_year;
```

,financial_year,trust_op_margin_pct,sector_median_op_margin_pct,trust_change_pts,trust_pay_pct,sector_median_pay_pct
0,2021/22,0.3,1.5,NaN,65.4,62.8
1,2022/23,2.0,0.1,1.7,67.7,64.7
2,2023/24,-22.9,-0.8,-24.9,69.7,64.8


## 3.5 Reported result against the result before impairments

Four SubCodes are pivoted out of the long fact table in one pass with conditional aggregation: income,
operating result and the impairment charge. Adding impairments back shows whether a deficit is a
day-to-day operating problem or a one-off, non-cash write-down.

In [6]:
nhs.sql("03_peer_benchmarking", "underlying_result")

`sql/analysis/03_peer_benchmarking.sql` → **underlying_result**

```sql
-- The trust's reported operating result next to its result before impairments,
-- every year. Impairments are pulled from the long fact table by SubCode.
-- Technique: pivot several SubCodes out of the EAV fact table in one pass with
-- conditional aggregation, then derive the underlying measures from them.
SELECT
    f.financial_year,
    ROUND(SUM(CASE WHEN f.sub_code IN ('SCI0100A', 'SCI0110A') THEN f.total_000s END) / 1000, 1) AS income_m,
    ROUND(SUM(CASE WHEN f.sub_code = 'SCI0140A' THEN f.total_000s END) / 1000, 1)               AS operating_result_m,
    ROUND(SUM(CASE WHEN f.sub_code = 'EXP0260'  THEN f.total_000s END) / 1000, 1)               AS impairments_m,
    ROUND((SUM(CASE WHEN f.sub_code = 'SCI0140A' THEN f.total_000s ELSE 0 END)
         + SUM(CASE WHEN f.sub_code = 'EXP0260'  THEN f.total_000s ELSE 0 END)) / 1000, 1)      AS result_before_impairments_m,
    ROUND(100 * (SUM(CASE WHEN f.sub_code = 'SCI0140A' THEN f.total_000s ELSE 0 END)
               + SUM(CASE WHEN f.sub_code = 'EXP0260'  THEN f.total_000s ELSE 0 END))
          / NULLIF(SUM(CASE WHEN f.sub_code IN ('SCI0100A', 'SCI0110A') THEN f.total_000s END), 0), 1)
                                                                                                  AS margin_before_impairments_pct
FROM nhs_silver.fct_tac f
WHERE f.org_code = @org_code
  AND f.sub_code IN ('SCI0100A', 'SCI0110A', 'SCI0140A', 'EXP0260')
GROUP BY f.financial_year
ORDER BY f.financial_year;
```

,financial_year,income_m,operating_result_m,impairments_m,result_before_impairments_m,margin_before_impairments_pct
0,2021/22,352.2,1.2,4.6,5.8,1.6
1,2022/23,383.6,7.6,-0.8,6.9,1.8
2,2023/24,397.3,-91.1,89.6,-1.5,-0.4
